# Homework 2 - Vector search

## Setting the environment 

In [ ]:
mkdir llm-zoomcamp-hw2 && cd llm-zoomcamp-hw2
uv init --no-workspace
uv add onnxruntime tokenizers numpy tqdm minsearch gitsource
uv add --dev huggingface-hub jupyter

# get two helper scripts
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed
wget $PREFIX/download.py
wget $PREFIX/embedder.py

#download.py fetches Xenova/all-MiniLM-L6-v2, the ONNX version of the all-MiniLM-L6-v2
uv run python download.py

## Embedding a query

In [4]:
from embedder import Embedder
embed = Embedder()

query1 = " How does approximate nearest neighbor search work?"
query1_v = embed.encode(query1)

#### Q1. The embedder returns a vector of 384 numbers. What's the first value v[0]?

In [10]:
print(f' The embedder returns {len(query1_v)} numbers.')
print(f' The first value is {query1_v[0]}')


 The embedder returns 384 numbers.
 The first value is -0.02058203437252893


## Loading the data

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]


{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Cosine similarity

#### Q2. What is the cosine similarity between query1 and the content of the page 02-vector-search/lessons/07-sqlitesearch-vector.md?

In [28]:
page = [doc for doc in documents if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"]
page_content = page[0]["content"]

page_content_v = embed.encode(page_content)

print(f" Cosine similarity between query1 and page conent is {page_content_v.dot(query1_v)}")


 Cosine similarity between query1 and page conent is 0.36107027225589694


## Chunking and search by hand

In [29]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [38]:
import numpy as np

contents = [chunk["content"] for chunk in chunks]
X = embed.encode_batch(contents)  
X = np.array(X)

q1_scores = X.dot(query1_v)

#### Q3. Which file does the highest-scoring chunk belong to?

In [43]:
print(f"The highest scoring filename is {chunks[np.argmax(q1_scores)]["filename"]}")


The highest scoring filename is 02-vector-search/lessons/07-sqlitesearch-vector.md


## Vector search with minsearch

In [50]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

query2 = "What metric do we use to evaluate a search engine?"
query2_v = embed.encode(query2)
first_result = vindex.search(query2_v)
first_result_filename = first_result[0]["filename"]

print(f"The filename of the first result is {first_result_filename}")

The filename of the first result is 04-evaluation/lessons/05-search-metrics.md


In [ ]:
query2 = "What metric do we use to evaluate a search engine?"